# Single Backtest Runner

This notebook orchestrates a single backtest run for the `ggTrader` engine. It supports flexible symbol selection (direct list or JSON file) and provides basic performance visualization.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if project_root not in sys.path:
    sys.path.append(project_root)

from ggTrader.utils.setup import load_data_and_setup
from ggTrader.core.trading import Trading
from ggTrader.utils.results_manager import ResultsManager

print("Environment initialized.")

In [ ]:
# --- Configuration ---
CONSTANTS = {
    "SYMBOLS": ["BTC", "ETH"],  # Set to None to use SYMBOLS_FILE
    "SYMBOLS_FILE": "data/top_50_consistent_movers.json",
    "INTERVAL": "4h",
    "START_DATE": "2024-01-01",
    "END_DATE": "2025-06-01",
    "START_CASH": 10000,
    "DEFAULT_PARAMS": {
        "adx_threshold": 25,
        "adx_length": 14,
        "sar_acceleration": 0.02,
        "sar_maximum": 0.2,
        "atr_multiplier": 3.0,
        "atr_length": 14,
        "use_dmp_cross": False,
    }
}

print("Configuration loaded.")

In [ ]:
print("Loading data...")
try:
    ohlcv = load_data_and_setup(CONSTANTS)
    print(f"Loaded {len(ohlcv)} rows for {len(ohlcv.columns.levels[0])} symbols.")
except Exception as e:
    print(f"Error loading data: {e}")

In [ ]:
print("Running backtest...")
engine = Trading(
    ohlcv_df=ohlcv,
    date_range=ohlcv.index,
    start_cash=CONSTANTS["START_CASH"],
    strategy_params=CONSTANTS["DEFAULT_PARAMS"],
)
engine.run()
print("Backtest complete.")

In [ ]:
print("Processing results...")
stats = engine.portfolio.stats_dict()
rm = ResultsManager("notebook_runner")

# Print summary
rm.print_summary(stats)

# Plot Equity Curve
fig = engine.portfolio.plot_value()
fig.show()